# Day 073 — Exercise 5: PodcastGenerator

**What you'll build:** `PodcastGenerator` — the full podcast pipeline with role-to-voice mapping, multi-segment synthesis, and file output.

**Why it matters:** The class abstracts the script format from the voice implementation. A script using role names (host, guest) works unchanged whether voices are English, French, or any other locale.

In [ ]:
import asyncio
from pathlib import Path
import asyncio
from pathlib import Path

COMMON_VOICES = [
    {'ShortName': 'en-US-AriaNeural',    'Gender': 'Female', 'Locale': 'en-US'},
    {'ShortName': 'en-US-GuyNeural',     'Gender': 'Male',   'Locale': 'en-US'},
    {'ShortName': 'en-GB-LibbyNeural',   'Gender': 'Female', 'Locale': 'en-GB'},
    {'ShortName': 'en-AU-NatashaNeural', 'Gender': 'Female', 'Locale': 'en-AU'},
    {'ShortName': 'fr-FR-DeniseNeural',  'Gender': 'Female', 'Locale': 'fr-FR'},
    {'ShortName': 'de-DE-KatjaNeural',   'Gender': 'Female', 'Locale': 'de-DE'},
]

DEFAULT_VOICE_MAP = {
    'host':     'en-US-AriaNeural',
    'guest':    'en-US-GuyNeural',
    'narrator': 'en-GB-LibbyNeural',
}

def build_prosody_ssml(text, rate='+0%', pitch='+0Hz', volume='+0%'):
    return (
        '<speak version="1.0" '
        'xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="en-US">'
        f'<prosody rate="{rate}" pitch="{pitch}" volume="{volume}">'
        f'{text}'
        '</prosody></speak>'
    )

def select_voice(voices, locale='en-US', gender=None):
    for v in voices:
        if v.get('Locale') != locale:
            continue
        if gender is not None and v.get('Gender', '').lower() != gender.lower():
            continue
        return v
    return None

def synthesize(text, voice='en-US-AriaNeural', tts_fn=None, rate='+0%', pitch='+0Hz'):
    if tts_fn is not None:
        return tts_fn(text, voice=voice, rate=rate, pitch=pitch)
    import edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

def synthesize_segments(segments, tts_fn=None):
    out = []
    for seg in segments:
        audio = synthesize(
            seg['text'],
            voice=seg.get('voice', 'en-US-AriaNeural'),
            tts_fn=tts_fn,
            rate=seg.get('rate', '+0%'),
            pitch=seg.get('pitch', '+0Hz'),
        )
        out.append(audio)
    return out
_mock_tts = lambda text, **kw: b'AUDIO:' + text[:12].encode()


## Task

Implement `PodcastGenerator`:

- `__init__(voice_map=None, tts_fn=None)`: store both; `self._voice_map = voice_map if voice_map is not None else DEFAULT_VOICE_MAP`
- `synthesize_segment(text, voice_key='host', rate='+0%', pitch='+0Hz') -> bytes`: `voice = self._voice_map.get(voice_key, DEFAULT_VOICE_MAP['host'])`; call `synthesize`
- `build(script) -> list[bytes]`: list comprehension calling `synthesize_segment` per entry using `e.get('voice_key','host')`, `e.get('rate','+0%')`, `e.get('pitch','+0Hz')`
- `save(script, output_dir) -> list[Path]`: `build` + mkdir + write `segment_NN.mp3` + return paths

## Your Implementation

In [ ]:
class PodcastGenerator:
    """Generate podcast-style audio from a script using Edge TTS.

    Inject tts_fn for testing without a network connection.
    """

    def __init__(self, voice_map=None, tts_fn=None) -> None:
        raise NotImplementedError

    def synthesize_segment(self, text: str, voice_key: str = 'host',
                            rate: str = '+0%', pitch: str = '+0Hz') -> bytes:
        """Synthesize one segment. Looks up voice from self._voice_map."""
        raise NotImplementedError

    def build(self, script: list) -> list:
        """Synthesize all script entries. Returns list[bytes].

        Each entry: {text, voice_key?, rate?, pitch?}
        """
        raise NotImplementedError

    def save(self, script: list, output_dir) -> list:
        """Build and save to segment_NN.mp3 files. Returns list[Path]."""
        raise NotImplementedError


In [ ]:
class PodcastGenerator:
    def __init__(self, voice_map=None, tts_fn=None):
        self._voice_map = voice_map if voice_map is not None else DEFAULT_VOICE_MAP
        self._tts_fn    = tts_fn

    def synthesize_segment(self, text, voice_key='host', rate='+0%', pitch='+0Hz'):
        voice = self._voice_map.get(voice_key, DEFAULT_VOICE_MAP['host'])
        return synthesize(text, voice=voice, tts_fn=self._tts_fn,
                          rate=rate, pitch=pitch)

    def build(self, script):
        return [
            self.synthesize_segment(
                e['text'],
                e.get('voice_key', 'host'),
                e.get('rate', '+0%'),
                e.get('pitch', '+0Hz'),
            )
            for e in script
        ]

    def save(self, script, output_dir):
        parts = self.build(script)
        out   = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        paths = []
        for i, audio in enumerate(parts, start=1):
            p = out / f'segment_{i:02d}.mp3'
            p.write_bytes(audio)
            paths.append(p)
        return paths


## Automated checks

In [ ]:

import tempfile
score, total = 0, 5
try:
    gen = PodcastGenerator(tts_fn=_mock_tts)

    # synthesize_segment returns bytes
    seg_audio = gen.synthesize_segment('Welcome to the show.', voice_key='host')
    assert isinstance(seg_audio, bytes) and len(seg_audio) > 0
    score += 1; print("✅ synthesize_segment returns non-empty bytes")

    # build returns list of correct length
    script = [
        {'text': 'Hello everyone.', 'voice_key': 'host'},
        {'text': 'Hi there!',        'voice_key': 'guest'},
        {'text': 'And we begin.',    'voice_key': 'narrator'},
    ]
    parts = gen.build(script)
    assert isinstance(parts, list) and len(parts) == 3
    assert all(isinstance(p, bytes) for p in parts)
    score += 1; print("✅ build returns list of 3 bytes objects")

    # different voice_keys → voice map used
    captured = {}
    def _cap_voice(text, **kw): captured['voice'] = kw.get('voice'); return b'x'
    gen2 = PodcastGenerator(tts_fn=_cap_voice)
    gen2.synthesize_segment('Test', voice_key='guest')
    assert captured.get('voice') == DEFAULT_VOICE_MAP['guest']
    score += 1; print("✅ voice_key correctly mapped to voice ShortName")

    # save writes files
    with tempfile.TemporaryDirectory() as tmpdir:
        paths = gen.save(script, tmpdir)
        assert len(paths) == 3
        names = [Path(p).name for p in paths]
        assert names == ['segment_01.mp3', 'segment_02.mp3', 'segment_03.mp3']
        assert all(Path(p).stat().st_size > 0 for p in paths)
    score += 1; print("✅ save creates segment_NN.mp3 files with content")

    # unknown voice_key falls back to host voice
    captured2 = {}
    def _cap2(text, **kw): captured2['voice'] = kw.get('voice'); return b'x'
    gen3 = PodcastGenerator(tts_fn=_cap2)
    gen3.synthesize_segment('Hi', voice_key='unknown_role_xyz')
    assert captured2.get('voice') == DEFAULT_VOICE_MAP['host']
    score += 1; print("✅ unknown voice_key falls back to host voice")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class PodcastGenerator:
    def __init__(self, voice_map=None, tts_fn=None):
        self._voice_map = voice_map if voice_map is not None else DEFAULT_VOICE_MAP
        self._tts_fn    = tts_fn

    def synthesize_segment(self, text, voice_key='host', rate='+0%', pitch='+0Hz'):
        voice = self._voice_map.get(voice_key, DEFAULT_VOICE_MAP['host'])
        return synthesize(text, voice=voice, tts_fn=self._tts_fn,
                          rate=rate, pitch=pitch)

    def build(self, script):
        return [
            self.synthesize_segment(
                e['text'],
                e.get('voice_key', 'host'),
                e.get('rate', '+0%'),
                e.get('pitch', '+0Hz'),
            )
            for e in script
        ]

    def save(self, script, output_dir):
        parts = self.build(script)
        out   = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        paths = []
        for i, audio in enumerate(parts, start=1):
            p = out / f'segment_{i:02d}.mp3'
            p.write_bytes(audio)
            paths.append(p)
        return paths
```

**Why `voice_map if voice_map is not None else DEFAULT_VOICE_MAP`?** The same `is not None` check used for template injection (Day 64) and style templates (Day 70). An explicit empty dict `{}` would mean no voices — don't fall back to defaults silently.

</details>